In [1]:
import gsam
import torch

In [13]:
from gsam import GSAM, LinearScheduler, CosineScheduler, ProportionScheduler, sam
from utility.log import Log
from utility.initialize import initialize
from utility.step_lr import StepLR
from model.wide_res_net import WideResNet
from model.smooth_cross_entropy import smooth_crossentropy
from data.cifar import Cifar

In [15]:
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# Transformations to apply to the data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load the CIFAR-10 dataset
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=32, shuffle=False, num_workers=2)

# Define the 3-layer perceptron model
class ThreeLayerPerceptron(nn.Module):
    def __init__(self):
        super(ThreeLayerPerceptron, self).__init__()
        self.fc1 = nn.Linear(32 * 32 * 3, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        x = x.view(-1, 32 * 32 * 3)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Initialize the model, loss function, and optimizer
num_epochs = 10

model = ThreeLayerPerceptron()
criterion = nn.CrossEntropyLoss()
base_optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

scheduler = CosineScheduler(T_max=num_epochs*len(trainset), max_value=0.01, min_value=0.0, optimizer=base_optimizer)
rho_scheduler = ProportionScheduler(pytorch_lr_scheduler=scheduler, max_lr=0.01, min_lr=0.0,
 max_value=2.0, min_value=2.0)
optimizer = GSAM(params=model.parameters(), base_optimizer=base_optimizer, 
                 model=model, gsam_alpha=0.4, rho_scheduler=rho_scheduler, adaptive=True)

# Training loop
train_loss_history = []

for epoch in range(nums_epochs):
        model.train()
        log.train(len_dataset=len(trainset))

        for batch in dataset.train:
            inputs, targets = (b.cuda() for b in batch)
            '''
            # first forward-backward step
            enable_running_stats(model)
            predictions = model(inputs)
            loss = smooth_crossentropy(predictions, targets, smoothing=args.label_smoothing)
            loss.mean().backward()
            optimizer.first_step(zero_grad=True)

            # second forward-backward step
            disable_running_stats(model)
            smooth_crossentropy(model(inputs), targets, smoothing=args.label_smoothing).mean().backward()
            optimizer.second_step(zero_grad=True)
            '''
            def loss_fn(predictions, targets):
                return smooth_crossentropy(predictions, targets, smoothing=0.0).mean()
 
            optimizer.set_closure(loss_fn, inputs, targets)
            predictions, loss = optimizer.step()
    
            with torch.no_grad():
                correct = torch.argmax(predictions.data, 1) == targets
                log(model, loss.cpu().repeat(32), correct.cpu(), scheduler.lr())
                scheduler.step()
                optimizer.update_rho_t()

        model.eval()
        log.eval(len_dataset=len(dataset.test))

        with torch.no_grad():
            for batch in dataset.test:
                inputs, targets = (b.cuda() for b in batch)

                predictions = model(inputs)
                loss = smooth_crossentropy(predictions, targets)
                correct = torch.argmax(predictions, 1) == targets
                log(model, loss.cpu(), correct.cpu())

    log.flush()

IndentationError: unindent does not match any outer indentation level (<tokenize>, line 94)